1 - From RGB, remove all that have no 

In [3]:
import arcpy
import os

# -------------------------------------------------------------------
# WHAT THIS SCRIPT DOES (4 lines)
# -------------------------------------------------------------------
# Converts LAS dataset (class 5 only) to multipoint features (requires point spacing).
# Explodes multipoints to single points.
# Spatially joins crowns to points to count points per crown.
# Exports crowns with Join_Count >= 1 to Crowns_RGB_cleaned.
# -------------------------------------------------------------------

arcpy.env.overwriteOutput = True

GDB = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"
CROWNS_IN  = os.path.join(GDB, "Crowns_RGB")
CROWNS_OUT = os.path.join(GDB, "Crowns_RGB_cleaned")
LASD = r"C:\ArcProj\AboveGroundBiomass\Wait_LasDataset.lasd"

scratch_gdb = arcpy.env.scratchGDB
LAS_MP  = os.path.join(scratch_gdb, "las_class5_mp_tmp")
LAS_PTS = os.path.join(scratch_gdb, "las_class5_pts_tmp")
SJ_OUT  = os.path.join(scratch_gdb, "crowns_pts_sj_class5_tmp")

for p in [LAS_MP, LAS_PTS, SJ_OUT, CROWNS_OUT]:
    if arcpy.Exists(p):
        arcpy.management.Delete(p)

# 1) LAS layer filtered to class 5
las_lyr = "lasd_lyr_class5"
if arcpy.Exists(las_lyr):
    arcpy.management.Delete(las_lyr)

arcpy.management.MakeLasDatasetLayer(
    in_las_dataset=LASD,
    out_layer=las_lyr,
    class_code=[5]
)

# 2) LAS -> multipoint (REQUIRES average_point_spacing)
# Set this to your approximate LAS point spacing in meters.
AVG_POINT_SPACING_M = 1.0

print("Converting LAS (class 5) to multipoint features...")
arcpy.ddd.LASToMultipoint(
    input=las_lyr,
    out_feature_class=LAS_MP,
    average_point_spacing=AVG_POINT_SPACING_M
)
print(f"Multipoints: {LAS_MP}")

# 3) Explode multipoints -> points
print("Exploding multipoints to points...")
arcpy.management.MultipartToSinglepart(LAS_MP, LAS_PTS)
print(f"Points: {LAS_PTS}")

# 4) Spatial join crowns <- points
print("Spatial joining crowns to class-5 points...")
arcpy.analysis.SpatialJoin(
    target_features=CROWNS_IN,
    join_features=LAS_PTS,
    out_feature_class=SJ_OUT,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT"
)

# 5) Keep crowns with >=1 point
print("Selecting crowns with at least 1 class-5 point...")
lyr = "crowns_sj_lyr"
arcpy.management.MakeFeatureLayer(SJ_OUT, lyr)
arcpy.management.SelectLayerByAttribute(lyr, "NEW_SELECTION", "Join_Count >= 1")

kept = int(arcpy.management.GetCount(lyr)[0])
print(f"Crowns kept: {kept}")

arcpy.management.CopyFeatures(lyr, CROWNS_OUT)
print(f"Output written: {CROWNS_OUT}")


Converting LAS (class 5) to multipoint features...
Multipoints: C:\ArcProj\AboveGroundBiomass\scratch.gdb\las_class5_mp_tmp
Exploding multipoints to points...
Points: C:\ArcProj\AboveGroundBiomass\scratch.gdb\las_class5_pts_tmp
Spatial joining crowns to class-5 points...
Selecting crowns with at least 1 class-5 point...
Crowns kept: 5484
Output written: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb\Crowns_RGB_cleaned


Remove overlaps

In [6]:
import arcpy
import os

arcpy.env.overwriteOutput = True

GDB = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"
IN_FC   = os.path.join(GDB, "Crowns_RGB_cleaned")
IN_2D   = os.path.join(GDB, "Crowns_RGB_cleaned_2D")
OUT_FC  = os.path.join(GDB, "Crowns_RGB_cleaned_nooverlap")

for fc in [IN_2D, OUT_FC]:
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)

# 1) Drop Z by exporting as 2D
# CopyFeatures often preserves Z; FeatureClassToFeatureClass typically writes 2D unless Z is explicitly carried.
arcpy.conversion.FeatureClassToFeatureClass(
    in_features=IN_FC,
    out_path=GDB,
    out_name=os.path.basename(IN_2D)
)

# If it STILL has Z (rare but possible), force it via a 2D polygon geometry rebuild
desc = arcpy.Describe(IN_2D)
if getattr(desc, "hasZ", False):
    tmp = os.path.join(arcpy.env.scratchGDB, "tmp_noz")
    if arcpy.Exists(tmp):
        arcpy.management.Delete(tmp)

    sr = desc.spatialReference
    arcpy.management.CreateFeatureclass(arcpy.env.scratchGDB, "tmp_noz", "POLYGON", spatial_reference=sr)

    # copy attributes schema
    fields = [f for f in arcpy.ListFields(IN_2D) if f.type not in ("OID", "Geometry")]
    for f in fields:
        arcpy.management.AddField(tmp, f.name, f.type, field_length=f.length)

    ins_fields = ["SHAPE@"] + [f.name for f in fields]
    with arcpy.da.SearchCursor(IN_2D, ins_fields) as sc, arcpy.da.InsertCursor(tmp, ins_fields) as ic:
        for row in sc:
            # rebuild as 2D by stripping z from vertices
            geom = row[0]
            parts2d = []
            for part in geom:
                ring = [arcpy.Point(p.X, p.Y) for p in part if p]
                parts2d.append(ring)
            geom2d = arcpy.Polygon(arcpy.Array([arcpy.Array(r) for r in parts2d]), sr)
            ic.insertRow((geom2d, *row[1:]))

    arcpy.management.Delete(IN_2D)
    arcpy.management.CopyFeatures(tmp, IN_2D)

# 2) Remove overlaps (method enums vary; GRID is accepted in your call, so keep it)
arcpy.analysis.RemoveOverlapMultiple(
    in_features=IN_2D,
    out_feature_class=OUT_FC,
    method="GRID",
    join_attributes="ALL"
)

print(f"✅ Wrote: {OUT_FC}")


✅ Wrote: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb\Crowns_RGB_cleaned_nooverlap


Bite Lidar with RGB

In [7]:
import arcpy
import os

arcpy.env.overwriteOutput = True

GDB = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"

RGB_NO = os.path.join(GDB, "Crowns_RGB_cleaned_nooverlap")  # erase features
LIDAR  = os.path.join(GDB, "Crowns_LiDAR")                  # <-- change to your LiDAR FC
OUT    = os.path.join(GDB, "Crowns_LiDAR_outside_RGB")

if arcpy.Exists(OUT):
    arcpy.management.Delete(OUT)

# Keep OUTSIDE RGB = Erase LiDAR by RGB
if hasattr(arcpy.analysis, "PairwiseErase"):
    arcpy.analysis.PairwiseErase(LIDAR, RGB_NO, OUT)
else:
    arcpy.analysis.Erase(LIDAR, RGB_NO, OUT)

print(f"✅ Wrote: {OUT}")


✅ Wrote: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb\Crowns_LiDAR_outside_RGB


Delete all LiDAR leftovers

In [ ]:
Check size of our lidar

In [8]:
import arcpy
fc = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb\Crowns_LiDAR_outside_RGB"  # change
perims = []
with arcpy.da.SearchCursor(fc, ["SHAPE@LENGTH"]) as cur:
    for (p,) in cur:
        if p is not None:
            perims.append(float(p))

print("Min perimeter (m):", min(perims))
print("P05 perimeter (m):", sorted(perims)[int(0.05*len(perims))])
print("Median perimeter (m):", sorted(perims)[int(0.50*len(perims))])


Min perimeter (m): 0.07747604812579276
P05 perimeter (m): 3.1044181959920145
Median perimeter (m): 14.35856759914939


Remove all bits under 5m perimeter

In [14]:
import arcpy, os, math

arcpy.env.overwriteOutput = True

GDB = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"
IN_FC  = os.path.join(GDB, "Crowns_LiDAR_outside_RGB")  # <- change if needed
OUT_FC = os.path.join(GDB, "Crowns_LiDAR_outside_RGB_nosliv")

# thresholds (tune)
MIN_MEAN_W_M = 2.5     # kill “thin” polygons (try 0.5–2.0)
MIN_COMPACT  = 0.5    # kill stringy shapes (try 0.01–0.05)
MIN_AREA_M2  = 5.0     # optional safety (try 1–10)

# copy first (don’t destroy original)
if arcpy.Exists(OUT_FC):
    arcpy.management.Delete(OUT_FC)
arcpy.management.CopyFeatures(IN_FC, OUT_FC)

# add fields if missing
def add_field_if_missing(fc, name, ftype):
    if name.upper() not in {f.name.upper() for f in arcpy.ListFields(fc)}:
        arcpy.management.AddField(fc, name, ftype)

add_field_if_missing(OUT_FC, "AREA_M2", "DOUBLE")
add_field_if_missing(OUT_FC, "PERIM_M", "DOUBLE")
add_field_if_missing(OUT_FC, "MEAN_W_M", "DOUBLE")
add_field_if_missing(OUT_FC, "COMPACT", "DOUBLE")

# calculate metrics
with arcpy.da.UpdateCursor(OUT_FC, ["SHAPE@AREA", "SHAPE@LENGTH", "AREA_M2", "PERIM_M", "MEAN_W_M", "COMPACT"]) as cur:
    for a, p, *_ in cur:
        a = float(a) if a else 0.0
        p = float(p) if p else 0.0
        mean_w = (2.0 * a / p) if p > 0 else 0.0
        compact = (4.0 * math.pi * a / (p * p)) if p > 0 else 0.0
        cur.updateRow((a, p, a, p, mean_w, compact))

# select slivers and delete them
lyr = "crown_lyr"
arcpy.management.MakeFeatureLayer(OUT_FC, lyr)

where = (
    f"(AREA_M2 < {MIN_AREA_M2} AND (MEAN_W_M < {MIN_MEAN_W_M} OR COMPACT < {MIN_COMPACT})) "
    f"OR (MEAN_W_M < {MIN_MEAN_W_M} AND COMPACT < {MIN_COMPACT})"
)

arcpy.management.SelectLayerByAttribute(lyr, "NEW_SELECTION", where)
to_delete = int(arcpy.management.GetCount(lyr)[0])
print("Slivers selected:", to_delete)

arcpy.management.DeleteFeatures(lyr)
print("✅ Cleaned output:", OUT_FC)


Slivers selected: 13511
✅ Cleaned output: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb\Crowns_LiDAR_outside_RGB_nosliv


Split LiDAR with compact <0.3

In [17]:
import arcpy, os
from arcpy.sa import *

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

GDB = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"
IN_FC = os.path.join(GDB, "Crowns_LiDAR_outside_RGB_nosliv")

OUT_SPLIT = os.path.join(GDB, "Crowns_LiDAR_outside_RGB_split_compact03")
CELL = 1.0  # <-- set to your raster/CHM resolution

# temp
scratch = arcpy.env.scratchGDB
sel_fc  = os.path.join(scratch, "crowns_sel_compact_tmp")
ras     = os.path.join(scratch, "crowns_sel_ras_tmp")
ras2    = os.path.join(scratch, "crowns_sel_ras_open_tmp")
rg      = os.path.join(scratch, "crowns_sel_rg_tmp")
poly    = os.path.join(scratch, "crowns_sel_poly_tmp")
other   = os.path.join(scratch, "crowns_other_tmp")

for p in [sel_fc, ras, ras2, rg, poly, other, OUT_SPLIT]:
    if arcpy.Exists(p):
        arcpy.management.Delete(p)

# 1) Split target set (compact < 0.3) vs others
arcpy.analysis.Select(IN_FC, sel_fc, "COMPACT < 0.3")
arcpy.analysis.Select(IN_FC, other,  "COMPACT >= 0.3")

# 2) Vector -> raster
# Use any constant field; create one if needed
const_field = "ONE"
if const_field.upper() not in {f.name.upper() for f in arcpy.ListFields(sel_fc)}:
    arcpy.management.AddField(sel_fc, const_field, "SHORT")
    arcpy.management.CalculateField(sel_fc, const_field, "1", "PYTHON3")

arcpy.env.cellSize = CELL
arcpy.env.snapRaster = None

ras_obj = arcpy.conversion.PolygonToRaster(
    in_features=sel_fc,
    value_field=const_field,
    out_rasterdataset=ras,
    cell_assignment="CELL_CENTER",
    priority_field="NONE",
    cellsize=CELL
)

# 3) Morphological opening to break thin necks:
# Shrink 1 cell then expand 1 cell (tune radius 1–2)
# This is the key “split” trick.
b = Raster(ras)
shrunk  = Shrink(b, 3, 1)   # (in_raster, number_cells, zone_values)
opened  = Expand(shrunk, 3, 1)
opened.save(ras2)

# 4) RegionGroup to label separated blobs
rg_obj = RegionGroup(Raster(ras2), "EIGHT", "WITHIN", "NO_LINK")
rg_obj.save(rg)

# 5) Raster -> polygons (now split)
arcpy.conversion.RasterToPolygon(
    in_raster=rg,
    out_polygon_features=poly,
    simplify="NO_SIMPLIFY",
    raster_field="VALUE"
)

# 6) Merge split polys back with untouched polys
arcpy.management.Merge([other, poly], OUT_SPLIT)

print(f"✅ Wrote: {OUT_SPLIT}")

✅ Wrote: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb\Crowns_LiDAR_outside_RGB_split_compact03


Combine the 2 datasets

In [18]:
import arcpy
import os

arcpy.env.overwriteOutput = True

GDB = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"

RGB   = os.path.join(GDB, "Crowns_RGB_cleaned_nooverlap")
LIDAR = os.path.join(GDB, "Crowns_LiDAR_outside_RGB_split_compact03")  # <-- your <0.3 split output
OUT   = os.path.join(GDB, "Crowns_RGB_LiDAR_merged")

# clean up
if arcpy.Exists(OUT):
    arcpy.management.Delete(OUT)

# add SOURCE field (recommended)
def ensure_source(fc, value):
    fields = {f.name.upper() for f in arcpy.ListFields(fc)}
    if "SOURCE" not in fields:
        arcpy.management.AddField(fc, "SOURCE", "TEXT", field_length=10)
    arcpy.management.CalculateField(fc, "SOURCE", f'"{value}"', "PYTHON3")

ensure_source(RGB, "RGB")
ensure_source(LIDAR, "LIDAR")

# merge
arcpy.management.Merge([RGB, LIDAR], OUT)

print(f"✅ Wrote merged crowns: {OUT}")


✅ Wrote merged crowns: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb\Crowns_RGB_LiDAR_merged


Summarise Pixels

In [20]:
import arcpy
import os
from arcpy.sa import ZonalStatisticsAsTable

# -------------------------------------------------------------------
# WHAT THIS SCRIPT DOES (4 lines)
# -------------------------------------------------------------------
# Adds an LC field to your merged crown polygons.
# Computes the MAJORITY land-cover class under each polygon using a class raster.
# Joins the majority result back to the polygons by OBJECTID.
# Writes LC = MAJORITY and removes the join field if you want.
# -------------------------------------------------------------------

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

GDB = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"
CROWNS = os.path.join(GDB, "Crowns_RGB_LiDAR_merged")

LCM = r"C:\Users\jserr\Dropbox\ABOVE GROUND BIOMASS ESTIMATES\England_3mLCM_v5.tif"  # <-- adjust path if different

# Output zonal table (scratch is fine)
ZONAL_TBL = os.path.join(arcpy.env.scratchGDB, "crowns_lcm_majority_tbl")

# Use OBJECTID as the zone field
zone_field = arcpy.Describe(CROWNS).OIDFieldName

# Make sure LC field exists
field_names = {f.name.upper() for f in arcpy.ListFields(CROWNS)}
if "LC" not in field_names:
    arcpy.management.AddField(CROWNS, "LC", "LONG")

# Clean old table
if arcpy.Exists(ZONAL_TBL):
    arcpy.management.Delete(ZONAL_TBL)

# (Recommended) ensure raster sampling is aligned
arcpy.env.snapRaster = LCM
arcpy.env.cellSize = LCM

# 1) Zonal majority
# ignore_nodata="DATA" means only real raster cells contribute (NoData ignored)
ZonalStatisticsAsTable(
    in_zone_data=CROWNS,
    zone_field=zone_field,
    in_value_raster=LCM,
    out_table=ZONAL_TBL,
    ignore_nodata="DATA",
    statistics_type="MAJORITY"
)

# 2) Join MAJORITY back to crowns
# Table will contain: zone_field, COUNT, AREA, MIN, MAX, RANGE, MEAN, STD, SUM, VARIETY, MAJORITY, MINORITY (varies)
arcpy.management.JoinField(
    in_data=CROWNS,
    in_field=zone_field,
    join_table=ZONAL_TBL,
    join_field=zone_field,
    fields=["MAJORITY"]
)

# 3) Copy MAJORITY -> LC (handle nulls)
arcpy.management.CalculateField(
    in_table=CROWNS,
    field="LC",
    expression="!MAJORITY! if !MAJORITY! is not None else None",
    expression_type="PYTHON3"
)

# Optional: drop the temporary joined MAJORITY field afterwards
# (If you want to keep it, comment these 2 lines out.)
arcpy.management.DeleteField(CROWNS, ["MAJORITY"])

print("✅ Done. LC populated from majority class in England_3mLCM_v5.tif")
print(f"Zonal table: {ZONAL_TBL}")


<expression>:1: SyntaxWarning: "is not" with 'int' literal. Did you mean "!="?


✅ Done. LC populated from majority class in England_3mLCM_v5.tif
Zonal table: C:\ArcProj\AboveGroundBiomass\scratch.gdb\crowns_lcm_majority_tbl


In [ ]:
If there are null LC do this

In [24]:
import arcpy, os
from arcpy.sa import ExtractValuesToPoints

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

GDB = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass2.gdb"
FC  = os.path.join(GDB, "Crowns_RGB_LiDAR_merged")
LCM = r"C:\Users\jserr\Dropbox\ABOVE GROUND BIOMASS ESTIMATES\England_3mLCM_v5.tif"  # <-- adjust path if different

OID = arcpy.Describe(FC).OIDFieldName

# Environments (helps alignment)
arcpy.env.snapRaster = LCM
arcpy.env.cellSize = LCM

# 0) Ensure stable ID + LC exist on the crowns
fields = {f.name.upper() for f in arcpy.ListFields(FC)}

if "CROWN_UID" not in fields:
    arcpy.management.AddField(FC, "CROWN_UID", "LONG")
    arcpy.management.CalculateField(FC, "CROWN_UID", f"!{OID}!", "PYTHON3")

if "LC" not in fields:
    arcpy.management.AddField(FC, "LC", "LONG")

# Temp outputs
scratch = arcpy.env.scratchGDB
nul_fc   = os.path.join(scratch, "crowns_lc_null_tmp")
pt_fc    = os.path.join(scratch, "crowns_lc_null_pts_tmp")
pt_val   = os.path.join(scratch, "crowns_lc_null_pts_val_tmp")

for p in [nul_fc, pt_fc, pt_val]:
    if arcpy.Exists(p):
        arcpy.management.Delete(p)

# 1) Export only the null LC polygons (keeps CROWN_UID)
arcpy.analysis.Select(FC, nul_fc, "LC IS NULL")

n_null = int(arcpy.management.GetCount(nul_fc)[0])
print("Null LC polygons:", n_null)
if n_null == 0:
    print("Nothing to fill.")
    raise SystemExit()

# 2) Create inside points (keeps CROWN_UID attribute)
arcpy.management.FeatureToPoint(nul_fc, pt_fc, "INSIDE")

# 3) Extract raster value to points
ExtractValuesToPoints(
    in_point_features=pt_fc,
    in_raster=LCM,
    out_point_features=pt_val,
    interpolate_values="NONE"
)

# Extracted field is usually RASTERVALU (sometimes RASTERVALU_1)
val_field = None
for cand in ["RASTERVALU", "RASTERVALU_1"]:
    if cand in {f.name.upper() for f in arcpy.ListFields(pt_val)}:
        # use the exact-cased field name
        for f in arcpy.ListFields(pt_val):
            if f.name.upper() == cand:
                val_field = f.name
                break
    if val_field:
        break

if not val_field:
    raise RuntimeError("Could not find extracted raster value field (expected RASTERVALU).")

# 4) Build dict: CROWN_UID -> LC value
uid2lc = {}
with arcpy.da.SearchCursor(pt_val, ["CROWN_UID", val_field]) as cur:
    for uid, v in cur:
        if uid is not None and v is not None:
            uid2lc[int(uid)] = int(v)

print("Centroid samples with value:", len(uid2lc))

# 5) Update crowns
updated = 0
with arcpy.da.UpdateCursor(FC, ["CROWN_UID", "LC"]) as ucur:
    for uid, lc in ucur:
        if lc is None and uid is not None:
            v = uid2lc.get(int(uid))
            if v is not None:
                ucur.updateRow((uid, v))
                updated += 1

print(f"✅ Filled LC for {updated} polygons.")


Null LC polygons: 876
Centroid samples with value: 876
✅ Filled LC for 876 polygons.
